# AutoScope: parasite box detectors

Trains **YOLO26** and **Faster R-CNN** to find malaria parasites, on the same
patient split every published number in this repo uses.

Runs on an **Azure ML compute instance**, a local GPU box, or Colab. It does not
clone anything and does not mount Drive: it finds the repo from wherever this
notebook sits, so on Azure ML you just pull the branch and open it.

## The rules this notebook enforces

1. **The split is inherited, not chosen.** `eval/results/wbc_sweep.json` ->
   `fit_patients` is 100 patient ids; the held-out 50 are the complement.
2. **Validation comes out of the fit patients**, 80/20 by patient.
3. **The held-out 50 are scored once**, at the bottom, at a threshold already
   decided. `detector_eval.py` refuses to sweep thresholds on them.

The number to beat, on those held-out patients: the patch network at its
shipped 0.75 threshold -- **precision 0.790, recall 0.739, F1 0.763, count
bias -6.4%**.

Quote the count bias next to the F1. The threshold moved from 0.80 to 0.75 not
because F1 improved (0.762 vs 0.763) but because 0.80 ran the count 14.4% low,
and the count is the denominator of every density a clinician reads.

## 1. Find the repo and the hardware

In [ ]:
import os, platform, subprocess, sys
from pathlib import Path

def find_repo(start=None):
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / 'pyproject.toml').exists() and (d / 'src' / 'autoscope').exists():
            return d
    raise SystemExit('Cannot find the repo root. Run this notebook from inside the checkout.')

ROOT = find_repo()
os.chdir(ROOT)
print('repo   :', ROOT)
print('python :', platform.python_version(), '|', platform.platform())

try:
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip() or 'nvidia-smi: no GPU listed')
except FileNotFoundError:
    print('nvidia-smi not found -- no NVIDIA driver on this machine')

## 2. Dependencies

`requirements.txt` + `requirements-detect.txt` resolve together on Linux with
**numpy 2.3.4** throughout. They did not always: `requirements-train.txt` used
to pin `numpy<2` for an Intel-Mac torch wheel, which could not coexist with
opencv-python 5 (`numpy>=2`). That is fixed, so all three install side by side.

If the Azure ML image already ships a different numpy, this pins it back. Restart
the kernel afterwards if pip says it upgraded numpy.

In [ ]:
INSTALL = True   # set False once the environment is built

if INSTALL:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',
                        '-r','requirements.txt','-r','requirements-detect.txt',
                        'matplotlib'], capture_output=True, text=True)
    print(r.stdout[-3000:] or 'installed')
    if r.returncode:
        print('STDERR:', r.stderr[-3000:])

In [ ]:
# The scripts carry a sys.path preamble, so no `pip install -e .` is needed --
# which is exactly what Azure ML does not do for a code snapshot.
sys.path[:0] = [str(ROOT / d) for d in ('src','eval','train')]

import json
import numpy, cv2, torch
from autoscope import annotations as ann, boxes as bx, parasites as par
import splits

ENV = {'python': platform.python_version(), 'numpy': numpy.__version__,
       'opencv': cv2.__version__, 'torch': torch.__version__,
       'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)
       if torch.cuda.is_available() else None}
print(json.dumps(ENV, indent=2))
(ROOT/'eval'/'results').mkdir(parents=True, exist_ok=True)
(ROOT/'eval'/'results'/'run_env.json').write_text(json.dumps(ENV, indent=2))
if not torch.cuda.is_available():
    print('\nWARNING: no CUDA. Export and visualisation are fine; training is not.')

## 3. The dataset

`data/download.py` refuses to start without ~12 GB free, because a half
download onto a full disk fails in a far more confusing way than a refusal.

In [ ]:
import shutil

RAW = ROOT/'data'/'raw'
free = shutil.disk_usage(ROOT).free
print(f'free disk: {free/1e9:.1f} GB')

have = RAW.exists() and any(p.name.startswith('TF') for p in RAW.glob('*'))
if not have:
    print('downloading the NLM set (3.9 GB)...')
    r = subprocess.run([sys.executable,'data/download.py'], cwd=str(ROOT))
    if r.returncode: raise SystemExit('download failed -- see the output above')
else:
    print('dataset already present')

root = splits.patient_root()
print('patient root:', root)
print('patients    :', len(splits.all_patients()))

### Verify the split before anything reads an image

In [ ]:
allp = splits.all_patients()
fit  = splits.fit_patients()
hold = splits.holdout_patients()
train_ids, val_ids = splits.train_val()

print(f'patients {len(allp)} | fit {len(fit & set(allp))} | holdout {len(hold)}')
print(f'train {len(train_ids)} | val {len(val_ids)}')
assert len(allp) == 150, f'expected 150 patients, found {len(allp)}'
assert len(hold) == 50,  f'expected 50 held out, found {len(hold)}'
assert not (set(train_ids) | set(val_ids)) & set(hold), 'held-out patient leaked into training'
print('\nsplit checks passed')

## 4. Look at the data before modelling it

Three things worth seeing with your own eyes, because each one has already
caused a bug in this project:

- the **annotations land on the cells** (the EXIF orientation trap),
- a parasite is genuinely **tiny** next to a white cell,
- the annotator's **circle is far bigger than the parasite**, which is why
  boxes are derived from the stain rather than drawn from the circle.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2 as cv

def pick_image(patient=None, min_parasites=20):
    """An annotated field with plenty of parasites, from a FIT patient."""
    pool = [patient] if patient else sorted(train_ids)
    for p in pool:
        for ip in sorted((root/p).glob('*.jpg')):
            if not ann.gt_path(ip).exists():
                continue
            img = cv.imread(str(ip))
            if img is None: continue
            h, w = img.shape[:2]
            t = ann.for_image(ip, w, h)
            if t and len(t.parasites) >= min_parasites:
                return ip, img, t
    raise SystemExit('no suitable image found')

IMG_PATH, IMG, TRUTH = pick_image()
print(IMG_PATH)
print(f'{IMG.shape[1]} x {IMG.shape[0]} px | '
      f'{len(TRUTH.parasites)} parasites | {len(TRUTH.white_cells)} white cells')

### The whole field, annotations overlaid

Parasites in red (the circle is the **annotator's**, not the object), white
cells in cyan. If the markers sat off the cells, the EXIF orientation flag
would be being read wrongly -- that is the failure this view catches instantly.

In [ ]:
def overlay(img, truth, scale=0.25, circles=True):
    vis = cv.cvtColor(img, cv.COLOR_BGR2RGB).copy()
    for a in truth.white_cells:
        cv.circle(vis, (int(a.x), int(a.y)), 46, (0, 220, 255), 6)
    for a in truth.parasites:
        if circles and a.radius:
            cv.circle(vis, (int(a.x), int(a.y)), int(a.radius), (255, 40, 40), 4)
        cv.drawMarker(vis, (int(a.x), int(a.y)), (255, 40, 40),
                      cv.MARKER_CROSS, 26, 4)
    return cv.resize(vis, None, fx=scale, fy=scale, interpolation=cv.INTER_AREA)

plt.figure(figsize=(9, 12))
plt.imshow(overlay(IMG, TRUTH))
plt.title(f'{IMG_PATH.parent.name}/{IMG_PATH.name}\n'
          f'red = parasite (circle is the annotator\'s), cyan = white cell', fontsize=10)
plt.axis('off'); plt.tight_layout(); plt.show()

### Native resolution, where the real problem shows

The same field, uncropped and unscaled. A parasite is about **10 px**; a white
cell nucleus is about **100**. This is why the export tiles at native
resolution: resize a 3000 px field to 640 and the parasite becomes 2 px, below
what any box head can regress.

The red circle is the annotator's. The green box is what `boxes.py` derives
from the stain. Measured across 953 parasites, the circle is about **3.3x**
the parasite across, so drawing boxes from it would make every target roughly
eleven times too large in area.

In [ ]:
prep, mask, ox, oy = par._prepare(IMG, True, True)
top, _, _ = par.tophat(prep)

# centre the crop on the densest clump of parasites
pts = np.array([[p.x - ox, p.y - oy] for p in TRUTH.parasites])
inside = pts[(pts[:,0] > 200) & (pts[:,1] > 200) &
             (pts[:,0] < prep.shape[1]-200) & (pts[:,1] < prep.shape[0]-200)]
cx, cy = (inside.mean(axis=0) if len(inside) else
          np.array([prep.shape[1]/2, prep.shape[0]/2])).astype(int)
R = 220
x0, y0 = max(cx-R, 0), max(cy-R, 0)
crop = prep[y0:y0+2*R, x0:x0+2*R].copy()
vis = cv.cvtColor(crop, cv.COLOR_BGR2RGB)

n_box = 0
for p in TRUTH.parasites:
    px, py = p.x - ox - x0, p.y - oy - y0
    if not (0 <= px < 2*R and 0 <= py < 2*R):
        continue
    if p.radius:
        cv.circle(vis, (int(px), int(py)), int(p.radius), (255, 60, 60), 2)
    b = bx.box_at(top, p.x - ox, p.y - oy, p.radius)
    if b is not None:
        cv.rectangle(vis, (int(b.x - x0), int(b.y - y0)),
                     (int(b.x + b.w - x0), int(b.y + b.h - y0)), (0, 230, 0), 2)
        n_box += 1

plt.figure(figsize=(11, 11))
plt.imshow(vis)
plt.title(f'{2*R} x {2*R} px at native resolution -- '
          f'red = annotator circle, green = derived box ({n_box} shown)', fontsize=11)
plt.axis('off'); plt.tight_layout(); plt.show()

### The size gap, as numbers

If the derived boxes were the same size as the annotator's circles, the ratio
below would be 1.0. It is nearer 0.3.

In [ ]:
sides, radii, ratios, dropped = [], [], [], 0
for p in TRUTH.parasites:
    b = bx.box_at(top, p.x - ox, p.y - oy, p.radius)
    if b is None:
        dropped += 1
        continue
    sides.append(max(b.w, b.h))
    if p.radius:
        radii.append(p.radius)
        ratios.append(max(b.w, b.h) / (2 * p.radius))

s, r, q = np.array(sides), np.array(radii), np.array(ratios)
print(f'parasites {len(TRUTH.parasites)} | boxed {len(s)} | dropped {dropped} '
      f'({100*dropped/max(len(TRUTH.parasites),1):.1f}%, no stain peak)')
print(f'derived box side px   : p10 {np.percentile(s,10):.0f}  median {np.median(s):.0f}  p90 {np.percentile(s,90):.0f}')
print(f'annotator radius px   : median {np.median(r):.0f}  -> circle diameter {2*np.median(r):.0f}')
print(f'box / circle diameter : median {np.median(q):.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(s, bins=20, color='#2a9d8f'); ax[0].set_title('derived box side (px)')
ax[0].axvline(np.median(s), color='k', ls='--', label=f'median {np.median(s):.0f}'); ax[0].legend()
ax[1].hist(2*r, bins=20, color='#e76f51'); ax[1].set_title("annotator circle diameter (px)")
ax[1].axvline(2*np.median(r), color='k', ls='--', label=f'median {2*np.median(r):.0f}'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Export the tiled dataset -- **the decision gate**

Fit patients only. Held-out patients are never read here.

**Read the drop rate before training anything.** It is the share of annotated
parasites for which no stain peak was found. Measured on 953 parasites it is
about 1%, which is fine; above ~35% the box route is fighting the data and a
point-native (heatmap) detector is the better bet.

Set `SMOKE = True` first. It exports a handful of patients in a couple of
minutes and proves the path before you commit half an hour.

In [ ]:
SMOKE = True

OUT = ROOT/'data'/'derived'/('detect_smoke' if SMOKE else 'detect')
cmd = [sys.executable, 'eval/export_detection.py', '--images-per-patient', '3',
       '--out', str(OUT)]
if SMOKE:
    cmd += ['--max-patients', '4', '--images-per-patient', '2']

print(' '.join(cmd), '\n')
r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])

In [ ]:
import json
m = json.loads((OUT/'manifest.json').read_text())
print(f"split source        {m['split_source']}")
print(f"train/val patients  {m['train_patients']} / {m['val_patients']}")
print(f"held out, untouched {m['holdout_patients_untouched']}")
print(f"tiles               train {m['counts'].get('tiles_train',0)}, val {m['counts'].get('tiles_val',0)}")
print(f"\nannotated parasites {m['annotated_parasites']}")
print(f"boxes derived       {m['boxes_derived']}")
print(f"DROP RATE           {m['box_drop_rate']*100:.1f}%    <- read this")
print(f"reached a tile      {m['boxes_in_tiles']} ({m['box_tile_loss_rate']*100:.1f}% lost to disc edge/tile bounds)")
print(f"box side px         p10 {m['box_side_p10_px']}, median {m['box_side_median_px']}, p90 {m['box_side_p90_px']}")

### The exported tiles, as the detector will see them

This is the last chance to catch a labelling bug before a GPU spends hours
learning it.

In [ ]:
import random

img_dir, lab_dir = OUT/'images'/'train', OUT/'labels'/'train'
withboxes = [p for p in sorted(img_dir.glob('*.jpg'))
             if (lab_dir/(p.stem+'.txt')).read_text().strip()]
print(f'{len(withboxes)} of {len(list(img_dir.glob("*.jpg")))} train tiles carry a box')

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, p in zip(axes, random.Random(0).sample(withboxes, min(3, len(withboxes)))):
    im = cv.cvtColor(cv.imread(str(p)), cv.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    n = 0
    for line in (lab_dir/(p.stem+'.txt')).read_text().splitlines():
        if not line.strip(): continue
        _, cx_, cy_, bw, bh = (float(v) for v in line.split())
        X0, Y0 = int((cx_-bw/2)*w), int((cy_-bh/2)*h)
        cv.rectangle(im, (X0, Y0), (X0+int(bw*w), Y0+int(bh*h)), (0, 230, 0), 2)
        n += 1
    ax.imshow(im); ax.set_title(f'{n} boxes', fontsize=9); ax.axis('off')
plt.suptitle('exported 640 px tiles with their YOLO labels', fontsize=11)
plt.tight_layout(); plt.show()

### Numeric check: are the labels actually on parasites?

An eyeball can miss a systematic few-pixel offset. The top-hat stain under the
box centres should be far above background.

In [ ]:
on, off = [], []
rng = np.random.default_rng(0)
for p in withboxes[:40]:
    im = cv.imread(str(p)); h, w = im.shape[:2]
    t_, _, _ = par.tophat(im)
    for line in (lab_dir/(p.stem+'.txt')).read_text().splitlines():
        if not line.strip(): continue
        _, cx_, cy_, _, _ = (float(v) for v in line.split())
        on.append(t_[int(cy_*h), int(cx_*w)])
    off.extend(t_[rng.integers(0,h,40), rng.integers(0,w,40)])

print(f'{len(on)} box centres checked')
print(f'top-hat stain at box centres : median {np.median(on):.1f}')
print(f'top-hat at random pixels     : median {np.median(off):.1f}')
print(f'-> boxes sit on {np.median(on)/max(np.median(off),0.01):.0f}x the stain of background')

## 6. Train YOLO26

Re-run the export with `SMOKE = False` first if you have only done the smoke
run. Augmentation is turned down on purpose: mosaic would halve the apparent
size of a 10 px parasite.

In [ ]:
EPOCHS_YOLO = 100

r = subprocess.run([sys.executable,'train/yolo26.py','--data',str(OUT),
                    '--epochs',str(EPOCHS_YOLO),'--batch','16','--no-clearml'],
                   cwd=str(ROOT))
print('exit', r.returncode)
YOLO_W = ROOT/'models'/'yolo26'/'yolo26'/'weights'/'best.pt'
print('weights:', YOLO_W, '|', 'present' if YOLO_W.exists() else 'MISSING')

## 7. Choose the threshold -- on VAL, never on holdout

`detector_eval.py` will refuse this sweep against the held-out split.

In [ ]:
r = subprocess.run([sys.executable,'eval/detector_eval.py','--backend','yolo',
                    '--weights',str(YOLO_W),'--split','val','--sweep',
                    '--per-patient','3'], cwd=str(ROOT))
print('exit', r.returncode)

## 8. Faster R-CNN, same tiles

Slower. Checkpoints are written every epoch, so an interruption costs one epoch.

In [ ]:
EPOCHS_FRCNN = 12

r = subprocess.run([sys.executable,'train/frcnn.py','--data',str(OUT),
                    '--epochs',str(EPOCHS_FRCNN),'--batch','4','--workers','2',
                    '--no-clearml'], cwd=str(ROOT))
print('exit', r.returncode)
ckpts = sorted((ROOT/'models'/'frcnn').glob('epoch*.pt'))
FRCNN_W = ckpts[-1] if ckpts else None
print('checkpoints:', len(ckpts), '| using', FRCNN_W)

In [ ]:
r = subprocess.run([sys.executable,'eval/detector_eval.py','--backend','frcnn',
                    '--weights',str(FRCNN_W),'--split','val','--sweep',
                    '--per-patient','3'], cwd=str(ROOT))
print('exit', r.returncode)

## 9. The held-out 50 -- run **once**

Everything above happened on fit patients. Set each threshold from its val
sweep **before** running these. Re-running with different thresholds until one
looks good is fitting on the held-out set, and it would invalidate the
generalisation claim the project rests on.

In [ ]:
THRESHOLD_YOLO = 0.25    # <- from the YOLO val sweep

r = subprocess.run([sys.executable,'eval/detector_eval.py','--backend','yolo',
                    '--weights',str(YOLO_W),'--split','holdout',
                    '--threshold',str(THRESHOLD_YOLO),'--per-patient','3',
                    '--tag','yolo26_holdout'], cwd=str(ROOT))
print('exit', r.returncode)

In [ ]:
THRESHOLD_FRCNN = 0.25   # <- from the Faster R-CNN val sweep

r = subprocess.run([sys.executable,'eval/detector_eval.py','--backend','frcnn',
                    '--weights',str(FRCNN_W),'--split','holdout',
                    '--threshold',str(THRESHOLD_FRCNN),'--per-patient','3',
                    '--tag','frcnn_holdout'], cwd=str(ROOT))
print('exit', r.returncode)

## 10. The comparison

In [ ]:
RES = ROOT/'eval'/'results'
print(f"{'model':<12}{'thr':>6}{'prec':>9}{'recall':>9}{'F1':>9}{'bias%':>10}")
for f in sorted(RES.glob('detector_eval_*holdout*.json')):
    d = json.loads(f.read_text())
    for thr, v in d['by_threshold'].items():
        print(f"{d['backend']:<12}{thr:>6}{v['precision']:>9.3f}"
              f"{v['recall']:>9.3f}{v['f1']:>9.3f}{v['count_bias_pct']:>+10.1f}")
print(f"{'patch CNN':<12}{'0.75':>6}{0.790:>9.3f}{0.739:>9.3f}{0.763:>9.3f}{-6.4:>+10.1f}   <- to beat")
print('\nA better F1 with a worse count bias is not a better detector:')
print('the count is the denominator of every density.')